<div dir="rtl">

# الدرس 51: كيف تُصنع النماذج اللغوية الكبيرة

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### حاسبة الحساب والذاكرة

</div>

In [ ]:
def train_flops(n_params, n_tokens):
    return 6 * n_params * n_tokens

def days(flops, tflops_per_gpu=100, n_gpus=1):
    return flops / (tflops_per_gpu * 1e12 * n_gpus) / 86400

models = {                                   # (parameters, training tokens)
    "our tiny GPT (lesson 49)": (809_856, 6_144_000),
    "hypothetical 1B, 20 tokens/param": (1e9, 20e9),
    "hypothetical 7B, 2T tokens": (7e9, 2e12),
    "hypothetical 70B, 15T tokens": (70e9, 15e12),
}
for name, (n, d) in models.items():
    c = train_flops(n, d)
    print(f"{name:34} {c:9.2e} FLOPs | 1 GPU: {days(c):12.4g} days | 1024 GPUs: {days(c, n_gpus=1024):9.3g} days")

In [ ]:
BYTES = {"fp32": 4, "bf16": 2, "int8": 1, "int4": 0.5}
for n in [0.5e9, 1.5e9, 3e9, 7e9]:
    row = " | ".join(f"{k} {n * b / 1e9:5.2f} GB" for k, b in BYTES.items())
    print(f"{n / 1e9:>4}B params: {row}")

<div dir="rtl">

### قالب المحادثة

</div>

In [ ]:
def to_chatml(messages, add_generation_prompt=True):
    text = ""
    for m in messages:
        text += f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
    if add_generation_prompt:
        text += "<|im_start|>assistant\n"            # the model continues from here
    return text

chat = [{"role": "system", "content": "أنت مساعد مفيد."},
        {"role": "user", "content": "ما عاصمة تركيا؟"}]
print(to_chatml(chat))

<div dir="rtl">

### الخسارة على الجواب فقط

</div>

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
vocab, prompt_len, answer_len = 50, 6, 4
tokens = torch.randint(0, vocab, (prompt_len + answer_len,))
labels = tokens.clone()
labels[:prompt_len] = -100                          # ignore the prompt tokens
logits = torch.randn(prompt_len + answer_len, vocab)

full = F.cross_entropy(logits, tokens)
answer_only = F.cross_entropy(logits, labels, ignore_index=-100)
manual = F.cross_entropy(logits[prompt_len:], tokens[prompt_len:])
print(f"all tokens {full:.3f} | answer only {answer_only:.3f} | answer tokens by hand {manual:.3f}")
print("tokens counted:", int((labels != -100).sum()))

<div dir="rtl">

### نموذج المكافأة وتحسين التفضيل المباشر

</div>

In [ ]:
def reward_model_loss(r_chosen, r_rejected):
    return -F.logsigmoid(r_chosen - r_rejected)

def dpo_loss(delta_chosen, delta_rejected, beta=0.1):
    """delta = log p_policy(answer) - log p_reference(answer)"""
    return -F.logsigmoid(beta * (delta_chosen - delta_rejected))

print("reward model:", round(reward_model_loss(torch.tensor(2.0), torch.tensor(0.5)).item(), 3))
print("DPO         :", round(dpo_loss(torch.tensor(2.0), torch.tensor(-1.0)).item(), 3))
for dc, dr in [(0.0, 0.0), (2.0, -1.0), (5.0, -5.0)]:
    print(f"chosen {dc:+}, rejected {dr:+} -> DPO loss {dpo_loss(torch.tensor(dc), torch.tensor(dr)).item():.3f}")

<div dir="rtl">

## تمرين: تكلفة نموذجك التالي

تريد تدريب نموذج بخمسين مليون معامل من الصفر، بالنسبة المثالية.

1. كم رمزاً تحتاج؟ وكم عملية حسابية؟
2. كم يوماً على كرت واحد يعطي فعلياً 10 تريليونات عملية في الثانية؟ هذا تقدير متفائل لكثير من كروت اللابتوب.
3. هل تتسع أوزانه للتدريب الكامل في كرت بذاكرة 4 غيغابايت؟ استخدم تقدير 16 بايتاً لكل معامل.

</div>

In [ ]:
# اكتب حلّك هنا
